In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

D:\Users\cccc\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
iris = load_iris()

X = iris.data
y = iris.target

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42,)

In [4]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [5]:
# Create labeled + unlabeled dataset

rng = np.random.RandomState(42)

labeled_mask = (rng.rand(len(y_train)) < 0.20)

y_semi = y_train.copy()

# -1 means unlabeled
y_semi[~labeled_mask] = -1


print("Total training:", len(X_train))
print("Labeled:", np.sum(labeled_mask))
print("Unlabeled:", np.sum(~labeled_mask))

Total training: 120
Labeled: 32
Unlabeled: 88


In [15]:
# Self-training

base_model = LogisticRegression(max_iter=1000)

self_training = SelfTrainingClassifier(base_model, threshold=0.90)

self_training.fit(X_train, y_semi)

,"estimator estimator: estimator objectAn estimator object implementing `fit` and `predict_proba`.Invoking the `fit` method will fit a clone of the passed estimator,which will be stored in the `estimator_` attribute... versionadded:: 1.6 `estimator` was added to replace `base_estimator`.",LogisticRegre...max_iter=1000)
,"threshold threshold: float, default=0.75The decision threshold for use with `criterion='threshold'`.Should be in [0, 1). When using the `'threshold'` criterion, a:ref:`well calibrated classifier <calibration>` should be used.",0.9
,"criterion criterion: {'threshold', 'k_best'}, default='threshold'The selection criterion used to select which labels to add to thetraining set. If `'threshold'`, pseudo-labels with predictionprobabilities above `threshold` are added to the dataset. If `'k_best'`,the `k_best` pseudo-labels with highest prediction probabilities areadded to the dataset. When using the 'threshold' criterion, a:ref:`well calibrated classifier <calibration>` should be used.",'threshold'
,"k_best k_best: int, default=10The amount of samples to add in each iteration. Only used when`criterion='k_best'`.",10
,"max_iter max_iter: int or None, default=10Maximum number of iterations allowed. Should be greater than or equalto 0. If it is `None`, the classifier will continue to predict labelsuntil no new pseudo-labels are added, or all unlabeled samples havebeen labeled.",10
,"verbose verbose: bool, default=FalseEnable verbose output.",False
Name,Type,Value
"classes_ classes_: ndarray or list of ndarray of shape (n_classes,)Class labels for each output. (Taken from the trained`estimator_`).","ndarray[int64](3,)","[0,1,2]"
estimator_ estimator_: estimator objectThe fitted estimator.,LogisticRegression,LogisticRegre...max_iter=1000)
"labeled_iter_ labeled_iter_: ndarray of shape (n_samples,)The iteration in which each sample was labeled. When a sample hasiteration 0, the sample was already labeled in the original dataset.When a sample has iteration -1, the sample was not labeled in anyiteration.","ndarray[int64](120,)","[ 1, 1,-1,..., 0,-1, 1]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,4


In [17]:
# Separate original labeled data

X_labeled = X_train[labeled_mask]
y_labeled = y_train[labeled_mask]

In [18]:
# Get originally unlabeled data

X_unlabeled = X_train[~labeled_mask]

In [19]:
# Predict pseudo-labels

pseudo_labels = self_training.predict(X_unlabeled)

probabilities = self_training.predict_proba(X_unlabeled)

In [21]:
# Calculate confidence

confidence = probabilities.max(axis=1)

In [22]:
confidence

array([0.99562391, 0.99866618, 0.50189158, 0.98351732, 0.98376318,
       0.99129459, 0.99820241, 0.45005577, 0.62213975, 0.99499322,
       0.64330489, 0.84233145, 0.49618582, 0.98790126, 0.58825097,
       0.46708448, 0.98630913, 0.97789902, 0.73891849, 0.98746021,
       0.99200511, 0.78801006, 0.95693406, 0.64024552, 0.97919431,
       0.57654842, 0.97644373, 0.56448128, 0.99826205, 0.80374287,
       0.80932455, 0.4738838 , 0.94062228, 0.71272678, 0.99336783,
       0.86978405, 0.97514194, 0.96938475, 0.78925549, 0.78409561,
       0.9521685 , 0.68676347, 0.71127877, 0.97033978, 0.66835801,
       0.63118629, 0.98571428, 0.69253301, 0.9738347 , 0.92237392,
       0.96438873, 0.62490374, 0.9629745 , 0.98726982, 0.94149972,
       0.97569392, 0.59749221, 0.7631395 , 0.59356885, 0.99452393,
       0.94053401, 0.47435928, 0.89355254, 0.75999361, 0.62229388,
       0.64945141, 0.49329383, 0.69147885, 0.99109256, 0.54700604,
       0.9401528 , 0.9417693 , 0.86117153, 0.99276519, 0.97329

In [23]:
# Select reliable pseudo-labels

threshold = 0.90

reliable_mask = (confidence >= threshold)

X_pseudo = X_unlabeled[reliable_mask]

y_pseudo = pseudo_labels[reliable_mask]

print("Reliable pseudo-labeled samples:", len(X_pseudo))

Reliable pseudo-labeled samples: 41


In [24]:
# Combine real + pseudo labels

X_final = np.concatenate([X_labeled,X_pseudo], axis=0)

In [25]:
y_final = np.concatenate([y_labeled,y_pseudo], axis=0)

In [27]:
print("Final training samples:", len(X_final))

Final training samples: 73


In [28]:
# Train YOUR final model

final_model = RandomForestClassifier(n_estimators=200,random_state=42)

final_model.fit(X_final,y_final)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstr

In [29]:
# Test final model

y_pred = final_model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Final Model Accuracy:", accuracy)

Final Model Accuracy: 1.0
